In [ ]:
"""
2_eda.ipynb
Author: Jacob Erasmus
Project: UP Honours Research
Purpose: Exploratory Data Analysis (EDA) for baseline statisical analysis and dimensionality prep.
Alignment with Methodology:
    - Strict data leakage prevention: Analyses only the 80% training set.
    - Anomaly profiling: Establishes the 'nu' parameter
"""


import pandas as pd
import numpy as np

print("--- Exploratory Data Analysis (Training Fold Only) ---")

####################################
# 1. LOAD ONLY THE 80% TRAINING DATA
####################################
# Only load the 80% training set as this guarantees zero data leakage from the 20% test vault into the baseline stats.
file_path = "logs_80percent.csv"
df_train = pd.read_csv(file_path)
print(f"Training Data Loaded Successfully. Shape: {df_train.shape[0]} rows, {df_train.shape[1]} columns.")

target_col = 'LabelEnc'
X_train = df_train.drop(columns=[target_col])
y_train = df_train[target_col]

######################################
# 2. EXACT CLASS IMBALANCE CALCULATION
######################################
print("\n--- 1. Class Imbalance Verification ---")
# Calculating exact percentages to mathematically justify the use of SMOTE and the setting of the 'nu' parameter in layer 1 of the hybrid ensemble.
class_counts = y_train.value_counts()
class_percentages = y_train.value_counts(normalize=True) * 100

imbalance_df = pd.DataFrame({
    'Count': class_counts,
    'Percentage (%)': class_percentages
})
print(imbalance_df)

###########################################
# 3. ZERO-VARIANCE DIMENSIONALITY REDUCTION
###########################################
print("\n--- 2. Zero-Variance Feature Identification ---")
# Find columns where there is only 1 unique value across the entire dataset, this feature provides no information gain and can be dropped.
unique_counts = X_train.nunique()
zero_variance_cols = unique_counts[unique_counts <= 1].index.tolist()

print(f"Found {len(zero_variance_cols)} columns with zero variance (no information gain).")
if len(zero_variance_cols) > 0:
    print("Sample of zero-variance columns to drop:", zero_variance_cols[:5])
    # Drop them immediately to save RAM and computation time later
    X_train = X_train.drop(columns=zero_variance_cols)
    print(f"Zero-variance columns dropped. New Feature Shape: {X_train.shape}")

###################################
# 4. SPARSITY / MISSING VALUE AUDIT
###################################
print("\n--- 3. Missing Value Audit ---")
missing_data = X_train.isnull().sum()
missing_data = missing_data[missing_data > 0].sort_values(ascending=False)

if missing_data.empty:
    print("Dataset is perfectly dense. No missing values detected.")
else:
    print(f"Found {len(missing_data)} columns with missing values:")
    print(missing_data.head())

--- Exploratory Data Analysis (Training Fold Only) ---


KeyboardInterrupt: 